## 🎯 Learning Objectives
* Design and implement a Convolutional Neural Network (CNN) architecture using PyTorch's `nn.Module`.
* Understand and apply common CNN layers such as `Conv2d`, `MaxPool2d`, and `Linear`.
* Configure and utilize PyTorch's data loading utilities (`Dataset`, `DataLoader`) for image classification tasks.
* Implement a complete training and evaluation loop for a deep learning model.
* Evaluate model performance using metrics like accuracy and loss on a validation set.


## DL03-L04: Exercise - Build a CNN from Scratch on CIFAR-10

### Task: Implement and Train a CNN for Image Classification

In this exercise, you will design, implement, and train a Convolutional Neural Network (CNN) from scratch using PyTorch to classify images from the CIFAR-10 dataset. The CIFAR-10 dataset consists of 60,000 32x32 color images in 10 classes, with 6,000 images per class. There are 50,000 training images and 10,000 test images.

This exercise will solidify your understanding of CNN architecture, PyTorch's `nn.Module` API, and the end-to-end deep learning workflow.

### Requirements:

1.  **Model Architecture**: Implement a custom CNN class inheriting from `torch.nn.Module`.
    *   Your CNN must include at least two `torch.nn.Conv2d` layers.
    *   It must include at least two `torch.nn.MaxPool2d` layers.
    *   It must include at least two `torch.nn.Linear` (fully connected) layers for classification.
    *   Use appropriate activation functions (e.g., `torch.nn.ReLU`) after convolutional and linear layers (except the final output layer).
    *   Ensure the input image dimensions (32x32x3 for CIFAR-10) are correctly handled through the layers.

2.  **Training Setup**: 
    *   Use `torch.optim.AdamW` as your optimizer.
    *   Use `torch.nn.CrossEntropyLoss` as your loss function.
    *   Train your model for a reasonable number of epochs (e.g., 5-10 epochs should be sufficient to see progress).
    *   Utilize a GPU if available (recommended for faster training).

3.  **Data Handling**: 
    *   Load the CIFAR-10 dataset using `torchvision.datasets.CIFAR10`.
    *   Apply standard transformations: convert to tensor and normalize the image data (mean and standard deviation for CIFAR-10 are approximately `(0.4914, 0.4822, 0.4465)` and `(0.2470, 0.2435, 0.2616)` respectively).
    *   Create `DataLoader` instances for both training and validation sets.

4.  **Training Loop**: 
    *   Implement a training loop that iterates through epochs and batches.
    *   Perform forward pass, calculate loss, backpropagation, and optimizer step.
    *   Track and print training loss and accuracy per epoch.

5.  **Evaluation Loop**: 
    *   Implement an evaluation loop to assess the model's performance on the validation set after each epoch.
    *   Calculate and print validation loss and accuracy.
    *   Ensure `model.eval()` and `torch.no_grad()` are used during evaluation.

### Evaluation Criteria:

*   **Correctness**: Does the code run without errors and correctly implement the required components?
*   **Architecture**: Does the CNN meet the minimum layer requirements?
*   **Performance**: Does the model show reasonable learning progress and achieve a non-trivial accuracy on the CIFAR-10 validation set (e.g., above 50% after a few epochs)?
*   **Code Quality**: Is the code well-structured, readable, and commented? Does it follow PyTorch best practices?
*   **Efficiency**: Does the code leverage GPU if available?


In [ ]:
# Standard library imports
import os
import time

# Third-party library imports
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

# --- Configuration and Hyperparameters ---
BATCH_SIZE = 128
LEARNING_RATE = 0.001
NUM_EPOCHS = 10 # Keep this low for quick execution during initial testing

# Device configuration
# Prioritize Apple Silicon's MPS, then CUDA, then fallback to CPU
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Using Apple Silicon MPS device.")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("Using NVIDIA CUDA device.")
else:
    DEVICE = torch.device("cpu")
    print("Using CPU device.")

# --- Data Loading and Preprocessing ---
# CIFAR-10 dataset statistics for normalization
# These values are standard for CIFAR-10 and help in faster convergence
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

# Define transformations for the training and validation sets
# Data augmentation (e.g., RandomCrop, RandomHorizontalFlip) is common for training
# but for this basic exercise, we'll keep it simple.
transform_train = transforms.Compose([
    transforms.ToTensor(), # Convert PIL Image to PyTorch Tensor
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD) # Normalize pixel values
])

transform_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

# Load CIFAR-10 datasets
# `download=True` will download the dataset if it's not already present
# `root='./data'` specifies where to store the downloaded data
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform_train
)

val_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform_val
)

# Create DataLoaders
# `shuffle=True` for training data is crucial for good generalization
# `num_workers` can speed up data loading, adjust based on your system's CPU cores
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=os.cpu_count() // 2 if os.cpu_count() else 0 # Use half available CPU cores
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False, # No need to shuffle validation data
    num_workers=os.cpu_count() // 2 if os.cpu_count() else 0
)

# Verify data shapes and classes
print(f"\nTraining dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

# Get a sample batch to check dimensions
sample_images, sample_labels = next(iter(train_loader))
print(f"Sample batch image shape: {sample_images.shape}") # Expected: [BATCH_SIZE, 3, 32, 32]
print(f"Sample batch label shape: {sample_labels.shape}") # Expected: [BATCH_SIZE]

# CIFAR-10 class names
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
print(f"CIFAR-10 classes: {classes}")


### Your Implementation Here

Now it's your turn! Based on the requirements outlined above, implement your CNN model, define the loss function and optimizer, and write the full training and evaluation loops. 

Feel free to experiment with different kernel sizes, number of filters, and fully connected layer sizes. Remember to move your model and data to the `DEVICE` (GPU/MPS/CPU) defined in the setup cell.

Good luck!


In [ ]:
# --- Reference Solution: CNN Model Definition ---
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        # First convolutional block
        # Input: 3 channels (RGB), Output: 32 channels, Kernel: 3x3
        # Padding='same' ensures output spatial dimensions match input for this kernel size
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding='same')
        self.relu1 = nn.ReLU()
        # Max pooling reduces spatial dimensions by half (32x32 -> 16x16)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Second convolutional block
        # Input: 32 channels, Output: 64 channels, Kernel: 3x3
        # Padding='same' ensures output spatial dimensions match input for this kernel size
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding='same')
        self.relu2 = nn.ReLU()
        # Max pooling reduces spatial dimensions by half (16x16 -> 8x8)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Third convolutional block (optional, but good for deeper features)
        # Input: 64 channels, Output: 128 channels, Kernel: 3x3
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding='same')
        self.relu3 = nn.ReLU()
        # Max pooling reduces spatial dimensions by half (8x8 -> 4x4)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully connected layers
        # After the last pooling layer (pool3), the feature map size is 4x4 with 128 channels.
        # So, the flattened size is 128 * 4 * 4 = 2048.
        self.fc1 = nn.Linear(128 * 4 * 4, 512) # First fully connected layer
        self.relu_fc1 = nn.ReLU()
        self.fc2 = nn.Linear(512, num_classes) # Output layer, 10 classes for CIFAR-10

    def forward(self, x):
        # Apply first conv -> relu -> pool
        x = self.pool1(self.relu1(self.conv1(x)))
        # Apply second conv -> relu -> pool
        x = self.pool2(self.relu2(self.conv2(x)))
        # Apply third conv -> relu -> pool
        x = self.pool3(self.relu3(self.conv3(x)))

        # Flatten the output for the fully connected layers
        # `x.view(-1, ...)` reshapes the tensor. -1 infers the batch size.
        x = x.view(-1, 128 * 4 * 4)

        # Apply fully connected layers
        x = self.relu_fc1(self.fc1(x))
        x = self.fc2(x) # No activation on the final layer for CrossEntropyLoss
        return x

# --- Instantiate Model, Loss, and Optimizer ---
model = SimpleCNN(num_classes=len(classes)).to(DEVICE)

# Loss function: CrossEntropyLoss is suitable for multi-class classification
criterion = nn.CrossEntropyLoss()

# Optimizer: AdamW is a robust choice, often outperforming Adam
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# --- Training and Evaluation Functions ---
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train() # Set the model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad() # Clear previous gradients
        loss.backward()       # Compute gradients
        optimizer.step()      # Update model parameters

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = 100 * correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

def evaluate_model(model, data_loader, criterion, device):
    model.eval() # Set the model to evaluation mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad(): # Disable gradient calculation for evaluation
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = 100 * correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

# --- Main Training Loop ---
print(f"\nStarting training for {NUM_EPOCHS} epochs...")
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = evaluate_model(model, val_loader, criterion, DEVICE)

    print(f"Epoch [{epoch}/{NUM_EPOCHS}] | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

end_time = time.time()
total_training_time = end_time - start_time
print(f"\nTraining finished in {total_training_time:.2f} seconds.")

# --- Final Evaluation on Validation Set ---
final_val_loss, final_val_acc = evaluate_model(model, val_loader, criterion, DEVICE)
print(f"\nFinal Validation Loss: {final_val_loss:.4f}")
print(f"Final Validation Accuracy: {final_val_acc:.2f}%")

# Optional: Save the trained model
# torch.save(model.state_dict(), 'cifar10_cnn_model.pth')
# print("Model saved to cifar10_cnn_model.pth")
